In [2]:
# 导入LangChain的字符串输出解析器，用于将AIMessage转换为字符串
from langchain_core.output_parsers import StrOutputParser
# 导入LangChain的聊天提示模板类
from langchain_core.prompts import ChatPromptTemplate
# 导入LangChain的OpenAI兼容接口（用于阿里云百炼等兼容OpenAI API的服务）
from langchain_openai import ChatOpenAI

# 创建翻译任务的提示词模板
# 系统消息：指定翻译任务，将英文翻译为目标语言
# 用户消息：提供待翻译的文本内容
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "Translate the following from English into {language}"),
    ("user", "{text}")
])

llm = ChatOpenAI(
    model="qwen-turbo",
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    openai_api_key="sk-4362e181dc44456fae7072e7eac4e970",
)

# 创建结果解析器
# StrOutputParser会将AIMessage对象转换为字符串（即提取content属性）
parser = StrOutputParser()

# 构建处理链：提示模板 → 大模型 → 结果解析器
# 使用管道操作符(|)连接各组件，形成完整的处理流程
chain = prompt_template | llm | parser

# 直接调用处理链，传入参数：待翻译文本和目标语言
# 这里将"nice to meet you"翻译为中文
print(chain.invoke({"text": "nice to meet you too", "language": "Chinese"}))


D:\python\PythonClass\venv\Lib\site-packages\langchain_core\_api\deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


你也好，很高兴见到你


In [3]:
analysis_prompt = ChatPromptTemplate.from_template("我应该怎么回答这句话？{talk}。给我一个五个字的示例")
chain2 = {"talk":chain}|analysis_prompt|llm|parser
print(chain2.invoke({"text":"nice to meet you", "language": "Chinese"}))

很高兴见到你。


In [4]:
# 从langchain_core.chat_history模块导入InMemoryChatMessageHistory类
from langchain_core.chat_history import InMemoryChatMessageHistory

# 创建InMemoryChatMessageHistory实例，用于在内存中存储聊天历史记录
history = InMemoryChatMessageHistory()

# 第一轮聊天：添加用户消息"你是谁？"
history.add_user_message("你是谁？")

# 调用大语言模型(LLM)生成回复，传入当前的聊天历史记录
aimessage = llm.invoke(history.messages)

# 打印AI生成的回复内容
print(aimessage.content)

# 将AI的回复添加到聊天历史中
history.add_message(aimessage)

# 第二轮聊天：添加一条消息"请重复一次"
# 注意：如果没有上一次聊天的记录，大模型可能不知道要重复什么内容
history.add_message("请重复一次")

# 再次调用大语言模型生成回复
aimessage2 = llm.invoke(history.messages)

# 打印第二次AI生成的回复内容
print(aimessage2.content)

# 将第二次AI的回复添加到聊天历史中
history.add_message(aimessage2)

# 打印历史聊天记录（当前被注释）
# print("Chat History:")
# for message in history.messages:
#     print(f"{type(message).__name__}: {message.content}")


你好！我是通义千问，是阿里巴巴集团旗下的通义实验室自主研发的超大规模语言模型。我能够回答问题、创作文字、逻辑推理、编程等，可以作为你的智能助手，帮助你完成各种任务。有什么我可以帮你的吗？
你好！我是通义千问，是阿里巴巴集团旗下的通义实验室自主研发的超大规模语言模型。我能够回答问题、创作文字、逻辑推理、编程等，可以作为你的智能助手，帮助你完成各种任务。有什么我可以帮你的吗？


In [ ]:
# 从langchain_redis模块导入RedisChatMessageHistory类
from langchain_redis import RedisChatMessageHistory

# 创建RedisChatMessageHistory实例，用于将聊天历史存储在Redis中
# session_id: 会话标识符，用于区分不同用户的聊天记录
# redis_url: Redis服务地址，本地需启动Redis服务
history = RedisChatMessageHistory(session_id="test", redis_url="redis://localhost:6379/0")

# 第一轮聊天：添加用户消息"你是谁？"
history.add_user_message("你是谁？")

# 调用大语言模型(LLM)生成回复
aimessage = llm.invoke(history.messages)

# 打印AI生成的回复内容
print(aimessage.content)

# 将AI的回复添加到聊天历史中
history.add_message(aimessage)

# 第二轮聊天：添加用户消息"请重复一次"
history.add_user_message("请重复一次")

# 再次调用大语言模型生成回复
aimessage2 = llm.invoke(history.messages)

# 打印第二次AI生成的回复内容
print(aimessage2.content)

# 将第二次AI的回复添加到聊天历史中
history.add_message(aimessage2)
